# SmartPort — Projet 01 : SQL, ETL, qualité et EDA (notebook unique, pas à pas)

**Avant de commencer**, importez dans Colab (icône dossier à gauche, puis *Importer*) uniquement les **5 CSV** :
`customers.csv`, `vessel_calls.csv`, `container_events.csv`, `yard_moves.csv`, `equipment.csv`.
Aucun autre fichier n'est nécessaire : tout le code est dans ce notebook.

**Deux parties :**
- **Partie A · SQL** : exploration des sources avec 18 requêtes (comptages, jointures, KPI, CTE, grain et clés).
- **Partie B · ETL, qualité et EDA** : pipeline SOURCES → RAW → STAGING → CORE → MART, rejets, contrôles qualité, métriques, puis graphiques et KPI.

**Comment travailler :** exécutez les cellules **dans l'ordre**, une par une (Maj + Entrée), et lisez le résultat avant de passer à la suivante.

## 0. La boîte à outils
Quelques fonctions que l'on réutilise tout au long du notebook. Prenez le temps de les lire : elles ne font rien d'autre que ce qui est expliqué au-dessus de chaque cellule.

### 0.1 Configuration
On importe les bibliothèques, on liste les 5 sources, et on prépare deux listes vides, `STEPS` et `CHECKS`, qui mémoriseront les étapes exécutées (elles serviront à **rejouer** le pipeline en section 7).
`REBUILT` liste les tables que l'on **reconstruit** à chaque exécution.

In [ ]:
import re, os, math, shutil, sqlite3, time
from datetime import datetime
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 60)

SOURCES = ["customers", "vessel_calls", "container_events", "yard_moves", "equipment"]

# Tables reconstruites à chaque exécution (le journal etl_run_metrics, lui, est conservé)
REBUILT = [
    "raw_customers", "raw_vessel_calls", "raw_container_events", "raw_yard_moves",
    "raw_equipment", "stg_customers", "stg_vessels", "stg_equipment",
    "stg_container_events", "stg_yard_moves", "fact_yard_move", "fact_container_event",
    "dim_equipment", "dim_freight_status", "dim_event_type", "dim_yard_area",
    "dim_vessel", "dim_customer", "dim_hour", "dim_date",
    "mart_kpi_events_per_day", "mart_kpi_move_by_area", "mart_kpi_vessel", "mart_kpi_service",
    "mart_kpi_customer", "mart_kpi_event_type", "mart_kpi_freight", "etl_rejects",
    "etl_quality_checks",
]

STEPS = []    # mémoire des étapes SQL exécutées
CHECKS = []   # mémoire des contrôles qualité exécutés

### 0.2 Connexion et journal
- `connect` ouvre la base SQLite et crée le **journal** `etl_run_metrics` (une ligne par étape : volume, durée, statut).
- `reset_layers` supprime toutes les tables reconstruites. C'est ce qui rend le pipeline **idempotent** : on repart toujours de zéro, sans risque de doublons.
- `new_run_id` donne un identifiant à l'exécution (la date et l'heure).
- `log_metric` écrit une ligne dans le journal.

In [ ]:
def connect(db_path="smartport_etl.db"):
    con = sqlite3.connect(db_path)
    con.execute("""CREATE TABLE IF NOT EXISTS etl_run_metrics (
        run_id TEXT, layer TEXT, step TEXT, rows_affected INTEGER,
        duration_s REAL, status TEXT, logged_at TEXT)""")
    con.commit()
    return con

def reset_layers(con):
    for t in REBUILT:
        con.execute(f"DROP TABLE IF EXISTS {t}")
    con.commit()

def new_run_id():
    return datetime.now().strftime("%Y%m%d-%H%M%S")

def log_metric(con, run_id, layer, step, rows, duration, status="OK"):
    con.execute("INSERT INTO etl_run_metrics VALUES (?,?,?,?,?,?,?)",
                (run_id, layer, step, rows, round(duration, 4), status,
                 datetime.now().isoformat(timespec="seconds")))
    con.commit()

### 0.3 Exécuter une étape
`run_step` exécute une requête SQL, **mesure sa durée et le nombre de lignes** produites, l'écrit dans le journal, et mémorise l'étape dans `STEPS`.
`load_raw` charge les 5 CSV dans la couche RAW. Tout est lu **en texte** et rien n'est transformé : on ajoute seulement deux colonnes de traçabilité (`_source_file`, `_loaded_at`).

In [ ]:
def run_step(con, run_id, layer, step, sql, register=True, verbose=True):
    t0 = time.perf_counter()
    before = con.total_changes
    con.execute(sql)
    con.commit()
    m = re.match(r"\s*CREATE TABLE (\w+) AS", sql, re.I)
    if m:                                    # CREATE TABLE ... AS SELECT : on compte les lignes créées
        rows = con.execute(f"SELECT COUNT(*) FROM {m.group(1)}").fetchone()[0]
    elif re.match(r"\s*(WITH|INSERT)", sql, re.I):
        rows = con.total_changes - before
    else:                                    # CREATE TABLE vide, DROP...
        rows = None
    dt = time.perf_counter() - t0
    log_metric(con, run_id, layer, step, rows, dt)
    if register and not any(s[1] == step for s in STEPS):
        STEPS.append((layer, step, sql))
    if verbose:
        print(f"[{layer:<7}] {step:<36} lignes={rows if rows is not None else '-':>6}   {dt*1000:7.1f} ms")
    return rows

def load_raw(con, run_id, data_dir=".", verbose=True):
    for s in SOURCES:
        t0 = time.perf_counter()
        df = pd.read_csv(f"{data_dir}/{s}.csv", dtype=str, keep_default_na=False)
        df["_source_file"] = f"{s}.csv"
        df["_loaded_at"] = datetime.now().isoformat(timespec="seconds")
        df.to_sql(f"raw_{s}", con, if_exists="replace", index=False)
        log_metric(con, run_id, "RAW", f"load raw_{s}", len(df), time.perf_counter() - t0)
        if verbose:
            print(f"[RAW    ] load raw_{s:<30} lignes={len(df):>6}")

### 0.4 Démarrage
On ouvre la base, on repart de zéro, et on crée `q()`, une petite fonction qui affiche le résultat d'un `SELECT` sous forme de tableau.

In [ ]:
con = connect("smartport_etl.db")
con.create_function("SQRT", 1, math.sqrt)   # SQRT n'existe pas toujours dans SQLite
reset_layers(con)
run_id = new_run_id()
print("run_id =", run_id)

def q(sql):
    return pd.read_sql(sql, con)

# PARTIE A · SQL : exploration des sources (étape 2)

Cette partie explore les **5 tables sources** avec du SQL, avant tout nettoyage. Objectif : comprendre les données, valider le **grain** (1 ligne = 1 événement, 1 ligne = 1 mouvement) et repérer les anomalies.

Elle répond aux 5 consignes du fichier `starter_queries.sql` du prof :

| Consigne du prof | Requêtes |
|---|---|
| Compter chaque table source | 1 |
| Au moins 3 jointures | 2, 3, 4 |
| Au moins 2 agrégations (KPI) | 5, 6, 7 |
| Une CTE | 8, 9 |
| Grain du DWH et unicité des clés | 10, 11, 12 (et 13 à 18 pour les KPI navires et clients) |

*Les requêtes sont numérotées de 1 à 18. Ne les confondez pas avec les contrôles qualité Q01 à Q15 de la partie B.*

Chaque requête est dans sa cellule, suivie d'une courte lecture du résultat.

### A.1 Création des tables sources
On utilise le schéma fourni par le prof (`schema_sources.sql`), recopié ici pour que le notebook n'ait besoin que des CSV.
La fonction `run` exécute un ordre SQL qui ne renvoie pas de tableau (`CREATE`, `DROP`...), contrairement à `q` qui affiche un `SELECT`.

In [ ]:
def run(sql):
    con.execute(sql)
    con.commit()
    print("OK")

In [ ]:
run("""DROP TABLE IF EXISTS customers;""")
run("""
CREATE TABLE customers (
    customer_id TEXT,
    customer_name TEXT,
    group_code TEXT,
    customer_type TEXT
);
""")

In [ ]:
run("""DROP TABLE IF EXISTS vessel_calls;""")
run("""
CREATE TABLE vessel_calls (
    vessel_id TEXT,
    vessel_name TEXT,
    vessel_type TEXT,
    service_code TEXT
);
""")

In [ ]:
run("""DROP TABLE IF EXISTS container_events;""")
run("""
CREATE TABLE container_events (
    event_id INTEGER,
    container_id TEXT,
    customer_id TEXT,
    vessel_id TEXT,
    event_type TEXT,
    event_ts TEXT,
    yard_area TEXT,
    freight_status TEXT,
    gross_weight_kg REAL
);
""")

In [ ]:
run("""DROP TABLE IF EXISTS yard_moves;""")
run("""
CREATE TABLE yard_moves (
    move_id INTEGER,
    container_id TEXT,
    move_ts TEXT,
    from_area TEXT,
    to_area TEXT,
    equipment_id TEXT
);
""")

In [ ]:
run("""DROP TABLE IF EXISTS equipment;""")
run("""
CREATE TABLE equipment (
    equipment_id TEXT,
    equipment_type TEXT,
    home_area TEXT
);
""")

### A.2 Chargement des CSV dans les tables sources
`if_exists="append"` ajoute les lignes dans les tables créées ci-dessus, en conservant les types du schéma (par exemple `gross_weight_kg` en nombre).

In [ ]:
for table in ["customers", "vessel_calls", "container_events", "yard_moves", "equipment"]:
    donnees = pd.read_csv(f"{table}.csv")
    donnees.to_sql(table, con, if_exists="append", index=False)
    print(table, len(donnees))

Vérification : le poids doit être de type nombre (`real`), sinon les moyennes seraient fausses.

In [ ]:
q("SELECT typeof(gross_weight_kg) AS type_poids, COUNT(*) AS nb FROM container_events GROUP BY 1")

### A.3 Les 18 requêtes

#### Comptage des tables

**Requête 1 · Comptage de chaque table source**

In [ ]:
q("""
SELECT 'customers' AS table_name, COUNT(*) AS nb_lignes FROM customers
UNION ALL SELECT 'vessel_calls', COUNT(*) FROM vessel_calls
UNION ALL SELECT 'container_events', COUNT(*) FROM container_events
UNION ALL SELECT 'yard_moves', COUNT(*) FROM yard_moves
UNION ALL SELECT 'equipment', COUNT(*) FROM equipment
""")

*Lecture :* Les 5 tables contiennent bien **80, 120, 9 000, 4 500 et 40 lignes** : l'import est complet et conforme à l'énoncé.

#### Jointures : événements × clients, événements × navires, mouvements × équipements

**Requête 2 · JOIN 1 : evenements x clients (activite par type de client)**

In [ ]:
q("""
SELECT c.customer_type, COUNT(*) AS nb_evenements,
       ROUND(AVG(e.gross_weight_kg),0) AS poids_moyen_kg
FROM container_events e
JOIN customers c ON c.customer_id = e.customer_id
GROUP BY c.customer_type
ORDER BY nb_evenements DESC
""")

*Lecture :* Les importateurs totalisent le plus d'événements (3 290), parce qu'ils sont les plus nombreux. Le poids moyen est quasi identique pour les 4 types de clients (18,5 à 18,7 tonnes).

**Requête 3 · JOIN 2 : evenements x navires (volumes par service)**

In [ ]:
q("""
SELECT v.service_code, COUNT(*) AS nb_evenements,
       COUNT(DISTINCT e.vessel_id) AS nb_navires,
       ROUND(SUM(e.gross_weight_kg)/1000.0,0) AS tonnes
FROM container_events e
JOIN vessel_calls v ON v.vessel_id = e.vessel_id
GROUP BY v.service_code
ORDER BY nb_evenements DESC
""")

*Lecture :* **GULF** domine (2 469 événements, 46 010 tonnes) devant REDSEA (1 959) et DJJIB (1 801). ASIA est le plus faible (1 221). Cette requête donne le total par *service* ; les requêtes 13 à 15 détaillent par navire.

**Requête 4 · JOIN 3 : mouvements de cour x equipements (charge par type d'equipement)**

In [ ]:
q("""
SELECT q.equipment_type, COUNT(DISTINCT q.equipment_id) AS nb_equipements,
       COUNT(*) AS nb_mouvements,
       ROUND(1.0*COUNT(*)/COUNT(DISTINCT q.equipment_id),1) AS mouvements_par_equipement
FROM yard_moves m
JOIN equipment q ON q.equipment_id = m.equipment_id
GROUP BY q.equipment_type
ORDER BY mouvements_par_equipement DESC
""")

*Lecture :* La charge par équipement est **homogène** : de 107,8 mouvements par équipement (RMG) à 123,5 (RS). Les 2 équipements RS sont les plus sollicités, mais l'écart est modeste (environ 14 % de plus que RMG) et ne repose que sur 2 machines : à ne pas surinterpréter.

#### Agrégations : les KPI de base

**Requête 5 · KPI 1 : evenements par jour (10 premiers jours les plus charges)**

In [ ]:
q("""
SELECT DATE(event_ts) AS jour, COUNT(*) AS nb_evenements
FROM container_events
GROUP BY DATE(event_ts)
ORDER BY nb_evenements DESC, jour
LIMIT 10
""")

*Lecture :* Le jour le plus chargé est le **20/06/2026** (50 événements), suivi du 30/05 (47) et du 11/06 (45).

**Requête 6 · KPI 2 : MOVE par zone Yard**

In [ ]:
q("""
SELECT yard_area, COUNT(*) AS nb_move
FROM container_events
WHERE event_type = 'MOVE'
GROUP BY yard_area
ORDER BY nb_move DESC
""")

*Lecture :* **Y04** est la zone la plus chargée (538 MOVE). Attention : `Y08B` est comptée ici comme une zone séparée (456). Rattachée à Y08, celle-ci atteindrait 946 : c'est le sujet de la standardisation dans la partie B.

**Requête 7 · KPI 5 + 6 : poids moyen par type d'evenement et part FCL**

In [ ]:
q("""
SELECT event_type, COUNT(*) AS nb,
       ROUND(AVG(gross_weight_kg),0) AS poids_moyen_kg,
       ROUND(100.0*SUM(freight_status='FCL')/COUNT(*),1) AS pct_fcl
FROM container_events
GROUP BY event_type
ORDER BY nb DESC
""")

*Lecture :* Le poids moyen est quasi identique pour tous les types d'événements (18,4 à 18,8 tonnes) et la part de conteneurs pleins reste entre 74,0 % et 77,0 %.

#### CTE et sous-requête

**Requête 8 · CTE : top 5 clients par activite avec leur part du total**

In [ ]:
q("""
WITH activite AS (
    SELECT customer_id, COUNT(*) AS nb_evenements,
           SUM(gross_weight_kg) AS poids_total
    FROM container_events
    GROUP BY customer_id
), total AS (SELECT SUM(nb_evenements) AS t FROM activite)
SELECT c.customer_name, c.customer_type, a.nb_evenements,
       ROUND(100.0*a.nb_evenements/t.t,2) AS pct_total
FROM activite a
JOIN customers c ON c.customer_id = a.customer_id
CROSS JOIN total t
ORDER BY a.nb_evenements DESC
LIMIT 5
""")

*Lecture :* Le premier client ne représente que **1,51 %** de l'activité : aucune dépendance à un gros client.

**Requête 9 · Sous-requete : jours dont l'activite depasse moyenne + 2 ecarts-types (pics)**

In [ ]:
q("""
WITH par_jour AS (
    SELECT DATE(event_ts) AS jour, COUNT(*) AS n FROM container_events GROUP BY 1
), stats AS (
    SELECT AVG(n) AS moy,
           SQRT(AVG(n*n) - AVG(n)*AVG(n)) AS ecart FROM par_jour
)
SELECT p.jour, p.n, ROUND(s.moy,1) AS moyenne
FROM par_jour p, stats s
WHERE p.n > s.moy + 2*s.ecart
ORDER BY p.n DESC;

-- ============ VALIDATION DU GRAIN ET DES CLES ============
""")

*Lecture :* **9 jours** dépassent la moyenne + 2 écarts-types (environ 41 événements). 7 d'entre eux tombent entre début mai et début juillet : cette observation est discutée dans l'EDA (partie B, section 9).

#### Grain, unicité des clés et intégrité référentielle

**Requête 10 · Unicite des cles primaires candidates (attendu : 0 doublon partout)**

In [ ]:
q("""
SELECT 'container_events.event_id' AS cle, COUNT(*) - COUNT(DISTINCT event_id) AS doublons FROM container_events
UNION ALL SELECT 'yard_moves.move_id', COUNT(*) - COUNT(DISTINCT move_id) FROM yard_moves
UNION ALL SELECT 'customers.customer_id', COUNT(*) - COUNT(DISTINCT customer_id) FROM customers
UNION ALL SELECT 'vessel_calls.vessel_id', COUNT(*) - COUNT(DISTINCT vessel_id) FROM vessel_calls
UNION ALL SELECT 'equipment.equipment_id', COUNT(*) - COUNT(DISTINCT equipment_id) FROM equipment
""")

*Lecture :* **Aucun doublon** : chaque clé est unique. Le grain « 1 ligne = 1 événement » (`event_id`) et « 1 ligne = 1 mouvement » (`move_id`) est validé.

**Requête 11 · Integrite referentielle (attendu : 0 orphelin partout)**

In [ ]:
q("""
SELECT 'events -> customers' AS controle, COUNT(*) AS orphelins FROM container_events e
  LEFT JOIN customers c ON c.customer_id=e.customer_id WHERE c.customer_id IS NULL
UNION ALL SELECT 'events -> vessel_calls', COUNT(*) FROM container_events e
  LEFT JOIN vessel_calls v ON v.vessel_id=e.vessel_id WHERE v.vessel_id IS NULL
UNION ALL SELECT 'yard_moves -> equipment', COUNT(*) FROM yard_moves m
  LEFT JOIN equipment q ON q.equipment_id=m.equipment_id WHERE q.equipment_id IS NULL
UNION ALL SELECT 'yard_moves -> events (container_id)', COUNT(*) FROM yard_moves m
  WHERE m.container_id NOT IN (SELECT container_id FROM container_events)
""")

*Lecture :* **Aucun orphelin** : tous les clients, navires et équipements référencés existent, et, dans cette version auditée des données, tous les conteneurs des mouvements existent dans les événements. La version précédente contenait 342 mouvements orphelins, corrigés par le prof.

**Requête 12 · Anomalies metier a surveiller**

In [ ]:
q("""
SELECT 'mouvements from_area = to_area' AS anomalie, COUNT(*) AS valeur FROM yard_moves WHERE from_area = to_area
UNION ALL SELECT 'evenements en zone Y08B', COUNT(*) FROM container_events WHERE yard_area = 'Y08B'
UNION ALL SELECT 'mouvements arrivant en zone Y08B (to_area)', COUNT(*) FROM yard_moves WHERE to_area = 'Y08B'
UNION ALL SELECT 'mouvements partant de la zone Y08B (from_area)', COUNT(*) FROM yard_moves WHERE from_area = 'Y08B'
""")

*Lecture :* **712 mouvements (15,8 %)** ont une zone de départ identique à la zone d'arrivée. `Y08B` apparaît dans 1 481 événements et 758 fois comme zone d'**arrivée**, jamais comme zone de départ : cela suggère une sous-zone de Y08, hypothèse à valider avec le propriétaire métier.

#### KPI complets : volumes par navire et par service, activité par client

**Requête 13 · KPI 3a : volumes par navire (top 10) avec son service**

In [ ]:
q("""
SELECT v.vessel_id, v.vessel_name, v.vessel_type, v.service_code,
       COUNT(*) AS nb_evenements,
       COUNT(DISTINCT e.container_id) AS nb_conteneurs,
       ROUND(SUM(e.gross_weight_kg)/1000.0,1) AS tonnes
FROM container_events e
JOIN vessel_calls v ON v.vessel_id = e.vessel_id
GROUP BY v.vessel_id, v.vessel_name, v.vessel_type, v.service_code
ORDER BY nb_evenements DESC
LIMIT 10
""")

*Lecture :* Les trois navires les plus actifs (V020, V039, V068) totalisent **100 événements** chacun.

**Requête 14 · KPI 3b : volumes par service (avec part du total)**

In [ ]:
q("""
SELECT v.service_code,
       COUNT(DISTINCT v.vessel_id) AS nb_navires,
       COUNT(*) AS nb_evenements,
       COUNT(DISTINCT e.container_id) AS nb_conteneurs,
       ROUND(SUM(e.gross_weight_kg)/1000.0,0) AS tonnes,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM container_events),1) AS pct_evenements
FROM container_events e
JOIN vessel_calls v ON v.vessel_id = e.vessel_id
GROUP BY v.service_code
ORDER BY nb_evenements DESC
""")

*Lecture :* GULF concentre 27,4 % des événements avec 32 navires. Par navire, les services sont comparables (voir l'EDA, partie B).

**Requête 15 · KPI 3c : volumes par type de navire (Container vs Feeder)**

In [ ]:
q("""
SELECT v.vessel_type, COUNT(DISTINCT v.vessel_id) AS nb_navires, COUNT(*) AS nb_evenements,
       ROUND(1.0*COUNT(*)/COUNT(DISTINCT v.vessel_id),1) AS evenements_par_navire
FROM container_events e
JOIN vessel_calls v ON v.vessel_id = e.vessel_id
GROUP BY v.vessel_type
""")

*Lecture :* Les navires Container traitent en moyenne 75,8 événements par navire, les Feeder 73,0 : l'écart est faible.

**Requête 16 · KPI 4a : activite par client (top 10)**

In [ ]:
q("""
SELECT c.customer_id, c.customer_name, c.customer_type, c.group_code,
       COUNT(*) AS nb_evenements,
       COUNT(DISTINCT e.container_id) AS nb_conteneurs,
       ROUND(SUM(e.gross_weight_kg)/1000.0,1) AS tonnes,
       ROUND(100.0*SUM(e.freight_status='FCL')/COUNT(*),1) AS pct_fcl
FROM container_events e
JOIN customers c ON c.customer_id = e.customer_id
GROUP BY c.customer_id, c.customer_name, c.customer_type, c.group_code
ORDER BY nb_evenements DESC
LIMIT 10
""")

*Lecture :* **Customer_067** (Forwarder) est le client le plus actif avec 136 événements, devant Customer_029 et Customer_077 (134 chacun).

**Requête 17 · KPI 4b : activite par groupe client**

In [ ]:
q("""
SELECT c.group_code, COUNT(DISTINCT c.customer_id) AS nb_clients, COUNT(*) AS nb_evenements,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM container_events),1) AS pct_evenements
FROM container_events e
JOIN customers c ON c.customer_id = e.customer_id
GROUP BY c.group_code
ORDER BY nb_evenements DESC
""")

*Lecture :* Les parts des groupes (ETH 32,9 %, DJ 28,6 %, FZ 22,3 %, ETH_MTL 16,2 %) sont proportionnelles à leur nombre de clients (26, 23, 18 et 13 sur 80).

**Requête 18 · KPI 4c : concentration (part cumulee des 10 premiers clients, via CTE)**

In [ ]:
q("""
WITH par_client AS (
    SELECT customer_id, COUNT(*) AS n FROM container_events GROUP BY customer_id
), top10 AS (
    SELECT n FROM par_client ORDER BY n DESC LIMIT 10
)
SELECT SUM(n) AS evenements_top10,
       ROUND(100.0*SUM(n)/(SELECT COUNT(*) FROM container_events),1) AS pct_top10,
       (SELECT MIN(n) FROM par_client) AS min_client,
       (SELECT MAX(n) FROM par_client) AS max_client
FROM top10
""")

*Lecture :* Les 10 premiers clients représentent **14,3 %** de l'activité (1 284 événements sur 9 000), et les clients vont de 91 à 136 événements : l'activité est très dispersée.

### A.4 Bilan de la partie A
- Les 5 sources sont **complètes** et leurs clés sont **uniques** : le grain est validé.
- Les données sont **cohérentes** : aucune clé orpheline, y compris entre mouvements et événements dans cette version auditée.
- Deux points d'attention sont repérés et traités dans la partie B : la zone **`Y08B`** (à rattacher ou non à Y08) et les **712 mouvements** dont la zone de départ est identique à la zone d'arrivée.
- Les valeurs sont **très homogènes** (poids, clients, équipements) : ce sera le fil conducteur de l'EDA.

Les requêtes de cette partie travaillent sur les données **brutes**. La partie B construit le pipeline qui les nettoie, les contrôle et les charge dans le Data Warehouse.

# PARTIE B · ETL, qualité, Data Warehouse et EDA

Cette partie construit le pipeline complet **SOURCES → RAW → STAGING → CORE → MART**, puis analyse les KPI. Elle reprend les mêmes données, mais en les **nettoyant, contrôlant et organisant**.

## 1. Couche RAW
Copie **fidèle** des CSV, sans aucune transformation. On garde la donnée d'origine intacte : si une règle de nettoyage est modifiée plus tard, on peut rejouer le pipeline depuis ici.

In [ ]:
load_raw(con, run_id)

In [ ]:
q('SELECT * FROM raw_container_events LIMIT 5')

## 2. Couche STAGING
On **nettoie et on contrôle** : suppression des espaces inutiles (`TRIM`), typage des dates et des nombres, standardisation de la zone (`Y08B` → `Y08`), puis **règles de rejet**.
Chaque ligne qui viole une règle est copiée dans `etl_rejects` avec l'identifiant de la règle, et n'entre pas dans le DWH.

Règles : `EV01` à `EV09` pour les événements, `MV01` à `MV06` pour les mouvements de cour (`MV04` et `MV06` correspondent aux règles `FK_EQUIPMENT` et `FK_CONTAINER` du prof).
Une anomalie n'est **pas** rejetée mais signalée plus loin (contrôle Q13) : les mouvements dont la zone de départ est identique à la zone d'arrivée. Les rejeter supprimerait 16 % des mouvements.

**stg_customers**

In [ ]:
run_step(con, run_id, "STAGING", "stg_customers", """
CREATE TABLE stg_customers AS
SELECT TRIM(customer_id) AS customer_id, TRIM(customer_name) AS customer_name,
       TRIM(group_code) AS group_code, TRIM(customer_type) AS customer_type
FROM raw_customers
WHERE TRIM(customer_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_customers GROUP BY TRIM(customer_id))
""")

**stg_vessels**

In [ ]:
run_step(con, run_id, "STAGING", "stg_vessels", """
CREATE TABLE stg_vessels AS
SELECT TRIM(vessel_id) AS vessel_id, TRIM(vessel_name) AS vessel_name,
       TRIM(vessel_type) AS vessel_type, TRIM(service_code) AS service_code
FROM raw_vessel_calls
WHERE TRIM(vessel_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_vessel_calls GROUP BY TRIM(vessel_id))
""")

**stg_equipment**

In [ ]:
run_step(con, run_id, "STAGING", "stg_equipment", """
CREATE TABLE stg_equipment AS
SELECT TRIM(equipment_id) AS equipment_id, TRIM(equipment_type) AS equipment_type,
       TRIM(home_area) AS home_area
FROM raw_equipment
WHERE TRIM(equipment_id) <> ''
  AND rowid IN (SELECT MIN(rowid) FROM raw_equipment GROUP BY TRIM(equipment_id))
""")

**create etl_rejects**

In [ ]:
run_step(con, run_id, "STAGING", "create etl_rejects", """
CREATE TABLE etl_rejects (
    reject_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_table TEXT, raw_rowid INTEGER, business_key TEXT,
    rule_id TEXT, rule_description TEXT)
""")

**EV01 event_id manquant**

In [ ]:
run_step(con, run_id, "STAGING", "EV01 event_id manquant", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV01', 'event_id manquant'
FROM raw_container_events WHERE event_id IS NULL OR TRIM(event_id) = ''
""")

**EV02 event_id en double**

In [ ]:
run_step(con, run_id, "STAGING", "EV02 event_id en double", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV02', 'event_id en double (on garde la 1re ligne)'
FROM raw_container_events WHERE TRIM(event_id) <> '' AND rowid NOT IN (SELECT MIN(rowid) FROM raw_container_events GROUP BY event_id)
""")

**EV03 date invalide**

In [ ]:
run_step(con, run_id, "STAGING", "EV03 date invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV03', 'event_ts illisible'
FROM raw_container_events WHERE datetime(event_ts) IS NULL
""")

**EV04 type hors domaine**

In [ ]:
run_step(con, run_id, "STAGING", "EV04 type hors domaine", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV04', 'event_type hors domaine'
FROM raw_container_events WHERE event_type NOT IN ('DISCHARGE','LOAD','MOVE','DELIVER','RECEIVE','STUFF','SEAL')
""")

**EV05 statut FCL/MTY invalide**

In [ ]:
run_step(con, run_id, "STAGING", "EV05 statut FCL/MTY invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV05', 'freight_status hors FCL/MTY'
FROM raw_container_events WHERE freight_status NOT IN ('FCL','MTY')
""")

**EV06 poids hors plage**

In [ ]:
run_step(con, run_id, "STAGING", "EV06 poids hors plage", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV06', 'poids non numerique ou hors 0..40000 kg'
FROM raw_container_events WHERE CAST(gross_weight_kg AS REAL) <= 0 OR CAST(gross_weight_kg AS REAL) > 40000
""")

**EV07 client inconnu**

In [ ]:
run_step(con, run_id, "STAGING", "EV07 client inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV07', 'customer_id absent du referentiel'
FROM raw_container_events WHERE customer_id NOT IN (SELECT customer_id FROM stg_customers)
""")

**EV08 navire inconnu**

In [ ]:
run_step(con, run_id, "STAGING", "EV08 navire inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV08', 'vessel_id absent du referentiel'
FROM raw_container_events WHERE vessel_id NOT IN (SELECT vessel_id FROM stg_vessels)
""")

**EV09 zone manquante**

In [ ]:
run_step(con, run_id, "STAGING", "EV09 zone manquante", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'container_events', rowid, event_id, 'EV09', 'yard_area manquante'
FROM raw_container_events WHERE yard_area IS NULL OR TRIM(yard_area) = ''
""")

**MV01 move_id manquant**

In [ ]:
run_step(con, run_id, "STAGING", "MV01 move_id manquant", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV01', 'move_id manquant'
FROM raw_yard_moves WHERE move_id IS NULL OR TRIM(move_id) = ''
""")

**MV02 move_id en double**

In [ ]:
run_step(con, run_id, "STAGING", "MV02 move_id en double", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV02', 'move_id en double (on garde la 1re ligne)'
FROM raw_yard_moves WHERE TRIM(move_id) <> '' AND rowid NOT IN (SELECT MIN(rowid) FROM raw_yard_moves GROUP BY move_id)
""")

**MV03 date invalide**

In [ ]:
run_step(con, run_id, "STAGING", "MV03 date invalide", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV03', 'move_ts illisible'
FROM raw_yard_moves WHERE datetime(move_ts) IS NULL
""")

**MV04 equipement inconnu**

In [ ]:
run_step(con, run_id, "STAGING", "MV04 equipement inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV04', 'equipment_id absent du referentiel'
FROM raw_yard_moves WHERE equipment_id NOT IN (SELECT equipment_id FROM stg_equipment)
""")

**MV05 zone manquante**

In [ ]:
run_step(con, run_id, "STAGING", "MV05 zone manquante", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV05', 'from_area ou to_area manquante'
FROM raw_yard_moves WHERE from_area IS NULL OR TRIM(from_area) = '' OR to_area IS NULL OR TRIM(to_area) = ''
""")

**MV06 conteneur inconnu**

In [ ]:
run_step(con, run_id, "STAGING", "MV06 conteneur inconnu", """
INSERT INTO etl_rejects (source_table, raw_rowid, business_key, rule_id, rule_description)
SELECT 'yard_moves', rowid, move_id, 'MV06', 'container_id absent de container_events'
FROM raw_yard_moves WHERE TRIM(container_id) NOT IN (SELECT TRIM(container_id) FROM raw_container_events)
""")

### Résultat des règles de rejet

In [ ]:
q('SELECT rule_id, rule_description, COUNT(*) AS nb FROM etl_rejects GROUP BY rule_id, rule_description')

*Un tableau vide signifie qu'aucune ligne n'est rejetée. C'est attendu sur ces données très propres : la section 8 prouve que les règles fonctionnent en injectant volontairement des erreurs.*

**stg_container_events**

In [ ]:
run_step(con, run_id, "STAGING", "stg_container_events", """
CREATE TABLE stg_container_events AS
SELECT rowid AS raw_rowid,
       CAST(event_id AS INTEGER)        AS event_id,
       TRIM(container_id)               AS container_id,
       TRIM(customer_id)                AS customer_id,
       TRIM(vessel_id)                  AS vessel_id,
       TRIM(event_type)                 AS event_type,
       datetime(event_ts)               AS event_ts,
       TRIM(yard_area)                  AS yard_area_raw,
       SUBSTR(TRIM(yard_area), 1, 3)    AS yard_area_std,   -- Y08B -> Y08
       TRIM(freight_status)             AS freight_status,
       CAST(gross_weight_kg AS REAL)    AS gross_weight_kg
FROM raw_container_events
WHERE rowid NOT IN (SELECT raw_rowid FROM etl_rejects WHERE source_table = 'container_events')
""")

**stg_yard_moves**

In [ ]:
run_step(con, run_id, "STAGING", "stg_yard_moves", """
CREATE TABLE stg_yard_moves AS
SELECT rowid AS raw_rowid,
       CAST(move_id AS INTEGER) AS move_id, TRIM(container_id) AS container_id,
       datetime(move_ts) AS move_ts, TRIM(from_area) AS from_area, TRIM(to_area) AS to_area,
       TRIM(equipment_id) AS equipment_id
FROM raw_yard_moves
WHERE rowid NOT IN (SELECT raw_rowid FROM etl_rejects WHERE source_table = 'yard_moves')
""")

In [ ]:
q('SELECT * FROM stg_container_events LIMIT 5')

`yard_area_std` contient la zone standardisée, `yard_area_raw` garde la zone d'origine. Comparaison :

In [ ]:
q("SELECT yard_area_raw, yard_area_std, COUNT(*) AS nb FROM stg_container_events GROUP BY 1, 2 ORDER BY 1")

## 3. Couche CORE (Data Warehouse en étoile)
### Le modèle
Le but du projet (comprendre les volumes, la nature des opérations et les pics d'activité, pour planifier les ressources) détermine les **faits** : ce que l'on veut compter. Les **dimensions** sont les axes selon lesquels on les regarde.

| Fait | Grain (1 ligne =) | Mesures | Question métier |
|---|---|---|---|
| `fact_container_event` | 1 événement conteneur | nombre d'événements, poids | Volumes, opérations, pics (les 6 KPI du sujet) |
| `fact_yard_move` | 1 mouvement de cour | nombre de mouvements, indicateur `is_same_area` | Utilisation de la cour et des équipements |

**Pourquoi deux faits ?** Les deux sources décrivent deux réalités différentes (une opération sur un conteneur, un déplacement dans la cour), avec des dimensions propres. Les fusionner créerait des colonnes vides et fausserait les comptages. Les dimensions `dim_date`, `dim_hour` et `dim_yard_area` sont **partagées** par les deux faits, ce qui permet de les comparer sur les mêmes axes.

`container_id`, `event_id` et `move_id` restent dans les faits comme *dimensions dégénérées* : il n'existe pas de table des conteneurs.

### La construction
Dimensions d'abord, puis faits. Les clés métier (`customer_id`...) sont remplacées par des **clés de substitution** (`customer_key`...).

**create dim_date**

In [ ]:
run_step(con, run_id, "CORE", "create dim_date", """
CREATE TABLE dim_date (date_key INTEGER PRIMARY KEY, full_date TEXT NOT NULL, year INTEGER, month INTEGER, day INTEGER,
    year_month TEXT, week_of_year INTEGER, day_of_week INTEGER, is_weekend INTEGER)
""")

**create dim_hour**

In [ ]:
run_step(con, run_id, "CORE", "create dim_hour", """
CREATE TABLE dim_hour (hour_key INTEGER PRIMARY KEY, period TEXT)
""")

**create dim_customer**

In [ ]:
run_step(con, run_id, "CORE", "create dim_customer", """
CREATE TABLE dim_customer (customer_key INTEGER PRIMARY KEY AUTOINCREMENT, customer_id TEXT NOT NULL UNIQUE,
    customer_name TEXT, group_code TEXT, customer_type TEXT)
""")

**create dim_vessel**

In [ ]:
run_step(con, run_id, "CORE", "create dim_vessel", """
CREATE TABLE dim_vessel (vessel_key INTEGER PRIMARY KEY AUTOINCREMENT, vessel_id TEXT NOT NULL UNIQUE,
    vessel_name TEXT, vessel_type TEXT, service_code TEXT)
""")

**create dim_yard_area**

In [ ]:
run_step(con, run_id, "CORE", "create dim_yard_area", """
CREATE TABLE dim_yard_area (area_key INTEGER PRIMARY KEY AUTOINCREMENT, area_code TEXT NOT NULL UNIQUE,
    parent_area TEXT NOT NULL, is_subarea INTEGER NOT NULL)
""")

**create dim_event_type**

In [ ]:
run_step(con, run_id, "CORE", "create dim_event_type", """
CREATE TABLE dim_event_type (event_type_key INTEGER PRIMARY KEY AUTOINCREMENT, event_type TEXT NOT NULL UNIQUE, event_family TEXT)
""")

**create dim_freight_status**

In [ ]:
run_step(con, run_id, "CORE", "create dim_freight_status", """
CREATE TABLE dim_freight_status (freight_key INTEGER PRIMARY KEY AUTOINCREMENT, freight_status TEXT NOT NULL UNIQUE, freight_label TEXT)
""")

**create dim_equipment**

In [ ]:
run_step(con, run_id, "CORE", "create dim_equipment", """
CREATE TABLE dim_equipment (equipment_key INTEGER PRIMARY KEY AUTOINCREMENT, equipment_id TEXT NOT NULL UNIQUE,
    equipment_type TEXT, home_area TEXT)
""")

**load dim_date**

In [ ]:
run_step(con, run_id, "CORE", "load dim_date", """
INSERT INTO dim_date
SELECT CAST(strftime('%Y%m%d', d) AS INTEGER), d, CAST(strftime('%Y', d) AS INTEGER), CAST(strftime('%m', d) AS INTEGER),
       CAST(strftime('%d', d) AS INTEGER), strftime('%Y-%m', d), CAST(strftime('%W', d) AS INTEGER),
       CAST(strftime('%w', d) AS INTEGER), CASE WHEN strftime('%w', d) IN ('0','6') THEN 1 ELSE 0 END
FROM (SELECT DISTINCT DATE(event_ts) AS d FROM stg_container_events
      UNION SELECT DISTINCT DATE(move_ts) FROM stg_yard_moves)
""")

**load dim_hour**

In [ ]:
run_step(con, run_id, "CORE", "load dim_hour", """
WITH RECURSIVE h(n) AS (SELECT 0 UNION ALL SELECT n+1 FROM h WHERE n < 23)
INSERT INTO dim_hour
SELECT n, CASE WHEN n < 6 THEN 'Nuit' WHEN n < 12 THEN 'Matin' WHEN n < 18 THEN 'Apres-midi' ELSE 'Soir' END FROM h
""")

**load dim_customer**

In [ ]:
run_step(con, run_id, "CORE", "load dim_customer", """
INSERT INTO dim_customer (customer_id, customer_name, group_code, customer_type)
SELECT customer_id, customer_name, group_code, customer_type FROM stg_customers ORDER BY customer_id
""")

**load dim_vessel**

In [ ]:
run_step(con, run_id, "CORE", "load dim_vessel", """
INSERT INTO dim_vessel (vessel_id, vessel_name, vessel_type, service_code)
SELECT vessel_id, vessel_name, vessel_type, service_code FROM stg_vessels ORDER BY vessel_id
""")

**load dim_yard_area**

In [ ]:
run_step(con, run_id, "CORE", "load dim_yard_area", """
INSERT INTO dim_yard_area (area_code, parent_area, is_subarea)
SELECT a, SUBSTR(a, 1, 3), CASE WHEN LENGTH(a) > 3 THEN 1 ELSE 0 END   -- regle : suffixe lettre = sous-zone
FROM (SELECT yard_area_raw AS a FROM stg_container_events
      UNION SELECT from_area FROM stg_yard_moves UNION SELECT to_area FROM stg_yard_moves) ORDER BY a
""")

**load dim_event_type**

In [ ]:
run_step(con, run_id, "CORE", "load dim_event_type", """
INSERT INTO dim_event_type (event_type, event_family)
SELECT DISTINCT event_type,
       CASE event_type WHEN 'DISCHARGE' THEN 'Operations navire' WHEN 'LOAD' THEN 'Operations navire'
                       WHEN 'RECEIVE' THEN 'Gate' WHEN 'DELIVER' THEN 'Gate' WHEN 'MOVE' THEN 'Cour'
                       WHEN 'STUFF' THEN 'Manutention' WHEN 'SEAL' THEN 'Manutention' ELSE 'Autre' END
FROM stg_container_events ORDER BY event_type
""")

**load dim_freight_status**

In [ ]:
run_step(con, run_id, "CORE", "load dim_freight_status", """
INSERT INTO dim_freight_status (freight_status, freight_label)
SELECT DISTINCT freight_status, CASE freight_status WHEN 'FCL' THEN 'Plein' WHEN 'MTY' THEN 'Vide' ELSE 'Inconnu' END
FROM stg_container_events ORDER BY freight_status
""")

**load dim_equipment**

In [ ]:
run_step(con, run_id, "CORE", "load dim_equipment", """
INSERT INTO dim_equipment (equipment_id, equipment_type, home_area)
SELECT equipment_id, equipment_type, home_area FROM stg_equipment ORDER BY equipment_id
""")

**create fact_container_event**

In [ ]:
run_step(con, run_id, "CORE", "create fact_container_event", """
CREATE TABLE fact_container_event (
    event_id INTEGER PRIMARY KEY, container_id TEXT NOT NULL,
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key), hour_key INTEGER NOT NULL REFERENCES dim_hour(hour_key),
    customer_key INTEGER NOT NULL REFERENCES dim_customer(customer_key), vessel_key INTEGER NOT NULL REFERENCES dim_vessel(vessel_key),
    area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key), event_type_key INTEGER NOT NULL REFERENCES dim_event_type(event_type_key),
    freight_key INTEGER NOT NULL REFERENCES dim_freight_status(freight_key),
    event_ts TEXT NOT NULL, event_count INTEGER NOT NULL DEFAULT 1, gross_weight_kg REAL)
""")

**create fact_yard_move**

In [ ]:
run_step(con, run_id, "CORE", "create fact_yard_move", """
CREATE TABLE fact_yard_move (
    move_id INTEGER PRIMARY KEY, container_id TEXT NOT NULL,
    date_key INTEGER NOT NULL REFERENCES dim_date(date_key), hour_key INTEGER NOT NULL REFERENCES dim_hour(hour_key),
    from_area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key), to_area_key INTEGER NOT NULL REFERENCES dim_yard_area(area_key),
    equipment_key INTEGER NOT NULL REFERENCES dim_equipment(equipment_key),
    move_ts TEXT NOT NULL, move_count INTEGER NOT NULL DEFAULT 1, is_same_area INTEGER NOT NULL)
""")

**load fact_container_event**

In [ ]:
run_step(con, run_id, "CORE", "load fact_container_event", """
INSERT INTO fact_container_event
SELECT e.event_id, e.container_id, CAST(strftime('%Y%m%d', e.event_ts) AS INTEGER), CAST(strftime('%H', e.event_ts) AS INTEGER),
       c.customer_key, v.vessel_key, a.area_key, t.event_type_key, f.freight_key, e.event_ts, 1, e.gross_weight_kg
FROM stg_container_events e
JOIN dim_customer c       ON c.customer_id = e.customer_id
JOIN dim_vessel v         ON v.vessel_id = e.vessel_id
JOIN dim_yard_area a      ON a.area_code = e.yard_area_raw
JOIN dim_event_type t     ON t.event_type = e.event_type
JOIN dim_freight_status f ON f.freight_status = e.freight_status
""")

**load fact_yard_move**

In [ ]:
run_step(con, run_id, "CORE", "load fact_yard_move", """
INSERT INTO fact_yard_move
SELECT m.move_id, m.container_id, CAST(strftime('%Y%m%d', m.move_ts) AS INTEGER), CAST(strftime('%H', m.move_ts) AS INTEGER),
       af.area_key, at.area_key, q.equipment_key, m.move_ts, 1, CASE WHEN m.from_area = m.to_area THEN 1 ELSE 0 END
FROM stg_yard_moves m
JOIN dim_yard_area af ON af.area_code = m.from_area
JOIN dim_yard_area at ON at.area_code = m.to_area
JOIN dim_equipment q  ON q.equipment_id = m.equipment_id
""")

Aperçu du fait principal, avec ses clés de substitution :

In [ ]:
q('SELECT * FROM fact_container_event LIMIT 5')

## 4. Couche MART (une table par KPI)
Tables prêtes à être lues par l'EDA ou par un outil de tableau de bord.

**mart_kpi_events_per_day**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_events_per_day", """
CREATE TABLE mart_kpi_events_per_day AS
SELECT d.full_date, d.year_month, COUNT(*) AS nb_evenements, COUNT(DISTINCT f.container_id) AS nb_conteneurs,
       ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes
FROM fact_container_event f JOIN dim_date d ON d.date_key = f.date_key
GROUP BY d.full_date, d.year_month
""")

In [ ]:
q("SELECT * FROM mart_kpi_events_per_day ORDER BY nb_evenements DESC LIMIT 5")

**mart_kpi_move_by_area**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_move_by_area", """
CREATE TABLE mart_kpi_move_by_area AS
SELECT a.area_code, a.parent_area, a.is_subarea, COUNT(*) AS nb_move
FROM fact_container_event f
JOIN dim_event_type t ON t.event_type_key = f.event_type_key
JOIN dim_yard_area a  ON a.area_key = f.area_key
WHERE t.event_type = 'MOVE'
GROUP BY a.area_code, a.parent_area, a.is_subarea
""")

In [ ]:
q("SELECT * FROM mart_kpi_move_by_area ORDER BY nb_move DESC")

**mart_kpi_vessel**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_vessel", """
CREATE TABLE mart_kpi_vessel AS
SELECT v.vessel_id, v.vessel_name, v.vessel_type, v.service_code, COUNT(*) AS nb_evenements,
       COUNT(DISTINCT f.container_id) AS nb_conteneurs, ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes
FROM fact_container_event f JOIN dim_vessel v ON v.vessel_key = f.vessel_key
GROUP BY v.vessel_id, v.vessel_name, v.vessel_type, v.service_code
""")

In [ ]:
q("SELECT * FROM mart_kpi_vessel ORDER BY nb_evenements DESC LIMIT 5")

**mart_kpi_service**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_service", """
CREATE TABLE mart_kpi_service AS
SELECT service_code, COUNT(*) AS nb_navires, SUM(nb_evenements) AS nb_evenements,
       ROUND(1.0*SUM(nb_evenements)/COUNT(*), 1) AS evenements_par_navire,
       ROUND(SUM(tonnes), 0) AS tonnes,
       ROUND(100.0*SUM(nb_evenements)/(SELECT SUM(nb_evenements) FROM mart_kpi_vessel), 1) AS pct_evenements
FROM mart_kpi_vessel GROUP BY service_code
""")

In [ ]:
q("SELECT * FROM mart_kpi_service ORDER BY nb_evenements DESC")

**mart_kpi_customer**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_customer", """
CREATE TABLE mart_kpi_customer AS
SELECT c.customer_id, c.customer_name, c.customer_type, c.group_code, COUNT(*) AS nb_evenements,
       COUNT(DISTINCT f.container_id) AS nb_conteneurs, ROUND(SUM(f.gross_weight_kg)/1000.0, 1) AS tonnes,
       ROUND(100.0*SUM(fs.freight_status = 'FCL')/COUNT(*), 1) AS pct_fcl
FROM fact_container_event f
JOIN dim_customer c        ON c.customer_key = f.customer_key
JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY c.customer_id, c.customer_name, c.customer_type, c.group_code
""")

In [ ]:
q("SELECT * FROM mart_kpi_customer ORDER BY nb_evenements DESC LIMIT 5")

**mart_kpi_event_type**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_event_type", """
CREATE TABLE mart_kpi_event_type AS
SELECT t.event_type, t.event_family, COUNT(*) AS nb_evenements, ROUND(AVG(f.gross_weight_kg), 0) AS poids_moyen_kg,
       ROUND(100.0*SUM(fs.freight_status = 'FCL')/COUNT(*), 1) AS pct_fcl
FROM fact_container_event f
JOIN dim_event_type t      ON t.event_type_key = f.event_type_key
JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY t.event_type, t.event_family
""")

In [ ]:
q("SELECT * FROM mart_kpi_event_type ORDER BY nb_evenements DESC")

**mart_kpi_freight**

In [ ]:
run_step(con, run_id, "MART", "mart_kpi_freight", """
CREATE TABLE mart_kpi_freight AS
SELECT fs.freight_status, fs.freight_label, COUNT(*) AS nb_evenements,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM fact_container_event), 1) AS pct_evenements,
       ROUND(AVG(f.gross_weight_kg), 0) AS poids_moyen_kg
FROM fact_container_event f JOIN dim_freight_status fs ON fs.freight_key = f.freight_key
GROUP BY fs.freight_status, fs.freight_label
""")

In [ ]:
q("SELECT * FROM mart_kpi_freight")

## 5. Contrôles qualité (15 contrôles du pipeline + les 17 règles du prof)
Chaque contrôle compare une valeur **observée** à une valeur **attendue**.
- `PASS` : conforme · `FAIL` : anomalie bloquante · `WARN` : anomalie métier signalée mais conservée · `INFO` : simple information.

### 5.1 La fonction `controle`
Elle exécute une requête, compare le résultat à la valeur attendue, affiche le statut, et l'enregistre dans la table `etl_quality_checks`.

In [ ]:
def init_quality_table(con):
    con.execute("DROP TABLE IF EXISTS etl_quality_checks")
    con.execute("""CREATE TABLE etl_quality_checks (check_id TEXT, layer TEXT, description TEXT,
                   observed REAL, expected REAL, status TEXT)""")
    con.commit()

def controle(con, check_id, layer, description, sql, expected=0, severity="FAIL", register=True, verbose=True):
    observed = con.execute(sql).fetchone()[0]
    status = "PASS" if observed == expected else severity
    con.execute("INSERT INTO etl_quality_checks VALUES (?,?,?,?,?,?)",
                (check_id, layer, description, observed, expected, status))
    con.commit()
    if register and not any(c[0] == check_id for c in CHECKS):
        CHECKS.append((check_id, layer, description, sql, expected, severity))
    if verbose:
        print(f"{status:<5} {check_id} [{layer}] {description} -> observé = {observed:g}")
    return status

init_quality_table(con)

### 5.2 Les contrôles, un par cellule

**Q01** · Volume : 1 si raw_container_events est vide (attendu 0)

In [ ]:
controle(con, "Q01", "RAW", "Volume : 1 si raw_container_events est vide (attendu 0)", """
SELECT CASE WHEN COUNT(*) > 0 THEN 0 ELSE 1 END FROM raw_container_events
""", 0, "FAIL")

**Q02** · Unicite de event_id dans fact_container_event

In [ ]:
controle(con, "Q02", "CORE", "Unicite de event_id dans fact_container_event", """
SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM fact_container_event
""", 0, "FAIL")

**Q03** · Unicite de move_id dans fact_yard_move

In [ ]:
controle(con, "Q03", "CORE", "Unicite de move_id dans fact_yard_move", """
SELECT COUNT(*) - COUNT(DISTINCT move_id) FROM fact_yard_move
""", 0, "FAIL")

**Q04** · Valeurs nulles dans les colonnes cles de stg_container_events

In [ ]:
controle(con, "Q04", "STAGING", "Valeurs nulles dans les colonnes cles de stg_container_events", """
SELECT COUNT(*) FROM stg_container_events WHERE event_id IS NULL OR container_id IS NULL OR customer_id IS NULL
     OR vessel_id IS NULL OR event_type IS NULL OR event_ts IS NULL OR yard_area_raw IS NULL OR gross_weight_kg IS NULL
""", 0, "FAIL")

**Q05** · Integrite referentielle : cles de fact_container_event absentes des dimensions

In [ ]:
controle(con, "Q05", "CORE", "Integrite referentielle : cles de fact_container_event absentes des dimensions", """
SELECT (SELECT COUNT(*) FROM fact_container_event WHERE customer_key NOT IN (SELECT customer_key FROM dim_customer))
          + (SELECT COUNT(*) FROM fact_container_event WHERE vessel_key NOT IN (SELECT vessel_key FROM dim_vessel))
          + (SELECT COUNT(*) FROM fact_container_event WHERE area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_container_event WHERE event_type_key NOT IN (SELECT event_type_key FROM dim_event_type))
          + (SELECT COUNT(*) FROM fact_container_event WHERE freight_key NOT IN (SELECT freight_key FROM dim_freight_status))
          + (SELECT COUNT(*) FROM fact_container_event WHERE date_key NOT IN (SELECT date_key FROM dim_date))
""", 0, "FAIL")

**Q06** · Integrite referentielle : cles de fact_yard_move absentes des dimensions

In [ ]:
controle(con, "Q06", "CORE", "Integrite referentielle : cles de fact_yard_move absentes des dimensions", """
SELECT (SELECT COUNT(*) FROM fact_yard_move WHERE from_area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE to_area_key NOT IN (SELECT area_key FROM dim_yard_area))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE equipment_key NOT IN (SELECT equipment_key FROM dim_equipment))
          + (SELECT COUNT(*) FROM fact_yard_move WHERE date_key NOT IN (SELECT date_key FROM dim_date))
""", 0, "FAIL")

**Q07** · Domaine des valeurs : event_type et freight_status hors referentiel

In [ ]:
controle(con, "Q07", "STAGING", "Domaine des valeurs : event_type et freight_status hors referentiel", """
SELECT COUNT(*) FROM stg_container_events
      WHERE event_type NOT IN ('DISCHARGE','LOAD','MOVE','DELIVER','RECEIVE','STUFF','SEAL') OR freight_status NOT IN ('FCL','MTY')
""", 0, "FAIL")

**Q08** · Plage du poids : valeurs hors 0..40000 kg dans fact_container_event

In [ ]:
controle(con, "Q08", "CORE", "Plage du poids : valeurs hors 0..40000 kg dans fact_container_event", """
SELECT COUNT(*) FROM fact_container_event WHERE gross_weight_kg <= 0 OR gross_weight_kg > 40000
""", 0, "FAIL")

**Q09** · Reconciliation evenements : RAW - (CORE + rejets)

In [ ]:
controle(con, "Q09", "CORE", "Reconciliation evenements : RAW - (CORE + rejets)", """
SELECT (SELECT COUNT(*) FROM raw_container_events) - (SELECT COUNT(*) FROM fact_container_event)
          - (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table = 'container_events')
""", 0, "FAIL")

**Q10** · Reconciliation mouvements : RAW - (CORE + rejets)

In [ ]:
controle(con, "Q10", "CORE", "Reconciliation mouvements : RAW - (CORE + rejets)", """
SELECT (SELECT COUNT(*) FROM raw_yard_moves) - (SELECT COUNT(*) FROM fact_yard_move)
          - (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table = 'yard_moves')
""", 0, "FAIL")

**Q11** · Conservation de la mesure : poids total STAGING - poids total CORE (kg)

In [ ]:
controle(con, "Q11", "CORE", "Conservation de la mesure : poids total STAGING - poids total CORE (kg)", """
SELECT ROUND((SELECT SUM(gross_weight_kg) FROM stg_container_events) - (SELECT SUM(gross_weight_kg) FROM fact_container_event), 3)
""", 0, "FAIL")

**Q12** · Coherence MART : somme des evenements par jour - lignes de fact_container_event

In [ ]:
controle(con, "Q12", "MART", "Coherence MART : somme des evenements par jour - lignes de fact_container_event", """
SELECT (SELECT SUM(nb_evenements) FROM mart_kpi_events_per_day) - (SELECT COUNT(*) FROM fact_container_event)
""", 0, "FAIL")

**Q13** · Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves)

In [ ]:
controle(con, "Q13", "CORE", "Coherence metier : mouvements de cour avec from_area = to_area (signales, conserves)", """
SELECT COUNT(*) FROM fact_yard_move WHERE is_same_area = 1
""", 0, "WARN")

**Q14** · Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER)

In [ ]:
controle(con, "Q14", "CORE", "Integrite : mouvements portant sur un conteneur absent des evenements (regle FK_CONTAINER)", """
SELECT COUNT(*) FROM fact_yard_move WHERE container_id NOT IN (SELECT container_id FROM fact_container_event)
""", 0, "FAIL")

**Q15** · Standardisation : evenements dont la zone a ete rattachee a sa zone parente (Y08B -> Y08)

In [ ]:
controle(con, "Q15", "STAGING", "Standardisation : evenements dont la zone a ete rattachee a sa zone parente (Y08B -> Y08)", """
SELECT COUNT(*) FROM stg_container_events WHERE yard_area_raw <> yard_area_std
""", 0, "INFO")

### 5.3 Les 17 règles qualité du prof
Le fichier `data_quality_rules.csv` demande, pour **chacune des 5 tables**, trois règles (`ROW_COUNT`, `DUPLICATE_CHECK`, `NULL_RATE`), plus deux règles de clés étrangères sur les mouvements de cour (`FK_CONTAINER`, `FK_EQUIPMENT`).
Ces contrôles portent sur la donnée **source (RAW)**, avant tout nettoyage. Gravité du prof : `HIGH` → `FAIL`, `MEDIUM` → `WARN`.

**ROW_COUNT** · la table n'est pas vide (gravité HIGH)

In [ ]:
TABLES_CLES = {"customers": "customer_id", "vessel_calls": "vessel_id", "container_events": "event_id",
              "yard_moves": "move_id", "equipment": "equipment_id"}

for t in TABLES_CLES:
    controle(con, f"DQ_ROW_COUNT_{t}", "RAW", f"ROW_COUNT {t} : table vide ? (0 = non vide)",
             f"SELECT CASE WHEN COUNT(*) > 0 THEN 0 ELSE 1 END FROM raw_{t}", 0, "FAIL")

Les volumes attendus, pour expliquer l'« évolution » demandée par la règle :

In [ ]:
q('''
SELECT 'customers' AS table_source, COUNT(*) AS nb_lignes FROM raw_customers
UNION ALL SELECT 'vessel_calls', COUNT(*) FROM raw_vessel_calls
UNION ALL SELECT 'container_events', COUNT(*) FROM raw_container_events
UNION ALL SELECT 'yard_moves', COUNT(*) FROM raw_yard_moves
UNION ALL SELECT 'equipment', COUNT(*) FROM raw_equipment
''')

**DUPLICATE_CHECK** · la clé métier est unique (gravité HIGH)

In [ ]:
for t, cle in TABLES_CLES.items():
    controle(con, f"DQ_DUPLICATE_CHECK_{t}", "RAW", f"DUPLICATE_CHECK {t} : doublons sur {cle}",
             f"SELECT COUNT(*) - COUNT(DISTINCT TRIM({cle})) FROM raw_{t}", 0, "FAIL")

**NULL_RATE** · pourcentage de cellules vides dans la table (gravité MEDIUM)

In [ ]:
for t in TABLES_CLES:
    cols = [r[1] for r in con.execute(f"PRAGMA table_info(raw_{t})") if not r[1].startswith("_")]
    vides = " + ".join(f"SUM(CASE WHEN {c} IS NULL OR TRIM({c}) = '' THEN 1 ELSE 0 END)" for c in cols)
    sql = f"SELECT ROUND(100.0 * ({vides}) / (COUNT(*) * {len(cols)}), 2) FROM raw_{t}"
    controle(con, f"DQ_NULL_RATE_{t}", "RAW", f"NULL_RATE {t} : % de cellules vides", sql, 0, "WARN")

**FK_CONTAINER et FK_EQUIPMENT** · les clés des mouvements existent dans les tables de référence (gravité HIGH)

In [ ]:
controle(con, "FK_CONTAINER", "RAW", "FK_CONTAINER : mouvements dont le container_id est absent de container_events",
         "SELECT COUNT(*) FROM raw_yard_moves WHERE TRIM(container_id) NOT IN (SELECT TRIM(container_id) FROM raw_container_events)", 0, "FAIL")

controle(con, "FK_EQUIPMENT", "RAW", "FK_EQUIPMENT : mouvements dont l'equipment_id est absent de equipment",
         "SELECT COUNT(*) FROM raw_yard_moves WHERE TRIM(equipment_id) NOT IN (SELECT TRIM(equipment_id) FROM raw_equipment)", 0, "FAIL")

### 5.4 Synthèse

In [ ]:
q('SELECT status, COUNT(*) AS nb FROM etl_quality_checks GROUP BY status ORDER BY status')

In [ ]:
q('SELECT * FROM etl_quality_checks')

### 5.5 Réconciliation
Pour chaque source : `lignes RAW = lignes chargées dans le CORE + lignes rejetées` (contrôles Q09 et Q10).

In [ ]:
q('''
SELECT 'container_events' AS source,
       (SELECT COUNT(*) FROM raw_container_events) AS raw,
       (SELECT COUNT(*) FROM fact_container_event) AS charge,
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='container_events') AS rejete
UNION ALL
SELECT 'yard_moves',
       (SELECT COUNT(*) FROM raw_yard_moves),
       (SELECT COUNT(*) FROM fact_yard_move),
       (SELECT COUNT(DISTINCT raw_rowid) FROM etl_rejects WHERE source_table='yard_moves')
''')

## 6. Monitoring
Le journal `etl_run_metrics` enregistre, pour chaque étape : volume, durée et statut. Il est **conservé** d'une exécution à l'autre.

In [ ]:
q(f"SELECT layer, step, rows_affected, ROUND(duration_s*1000, 1) AS duree_ms, status FROM etl_run_metrics WHERE run_id = '{run_id}' AND layer <> 'RAW'")

Durée et volume par couche :

In [ ]:
q(f"SELECT layer, COUNT(*) AS nb_etapes, ROUND(SUM(duration_s), 3) AS duree_s FROM etl_run_metrics WHERE run_id = '{run_id}' GROUP BY layer")

## 7. Idempotence
**Idempotent** veut dire : relancer le pipeline donne le même résultat, sans doublons.

### 7.1 Rejouer tout le pipeline
Pendant ce notebook, `STEPS` et `CHECKS` ont mémorisé toutes les étapes. La fonction `rejouer_pipeline` les **rejoue** d'un coup sur une base (sans rien afficher).

In [ ]:
def rejouer_pipeline(data_dir=".", db_path="smartport_etl.db"):
    c = connect(db_path)
    reset_layers(c)
    rid = new_run_id()
    load_raw(c, rid, data_dir, verbose=False)
    for layer, name, sql in STEPS:
        run_step(c, rid, layer, name, sql, register=False, verbose=False)
    init_quality_table(c)
    statuts = [controle(c, *chk, register=False, verbose=False) for chk in CHECKS]
    c.close()
    return statuts

print(len(STEPS), "étapes SQL et", len(CHECKS), "contrôles mémorisés")

### 7.2 Test : deux exécutions, même résultat ?

In [ ]:
def etat():
    return {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            for t in ["fact_container_event", "fact_yard_move", "dim_customer", "etl_rejects"]}

avant = etat()
runs_avant = con.execute("SELECT COUNT(DISTINCT run_id) FROM etl_run_metrics").fetchone()[0]
time.sleep(1.1)                                   # pour obtenir un run_id différent
rejouer_pipeline(".", "smartport_etl.db")
apres = etat()
runs_apres = con.execute("SELECT COUNT(DISTINCT run_id) FROM etl_run_metrics").fetchone()[0]

print("Avant :", avant)
print("Après :", apres)
print("Résultat identique :", avant == apres)
print(f"Exécutions dans le journal : {runs_avant} -> {runs_apres}")

## 8. Preuve que les règles de rejet fonctionnent
Les données du prof sont propres, donc `etl_rejects` est vide. Pour **prouver** que les règles marchent, on fabrique une copie des CSV à laquelle on ajoute **13 lignes volontairement défectueuses**, puis on rejoue le pipeline dessus (base séparée `smartport_test.db`).

In [ ]:
os.makedirs("bad", exist_ok=True)
for s in SOURCES:
    shutil.copy(f"{s}.csv", f"bad/{s}.csv")

e = pd.read_csv("bad/container_events.csv")
m = pd.read_csv("bad/yard_moves.csv")
b = e.iloc[0].to_dict()
def ev(**kw): d = dict(b); d.update(kw); return d

mauvais_events = [
    ev(event_id=e.event_id.iloc[1]),                 # EV02 : identifiant en double
    ev(event_id=99001, event_ts="pas-une-date"),     # EV03 : date illisible
    ev(event_id=99002, event_type="TELEPORT"),       # EV04 : type inconnu
    ev(event_id=99003, freight_status="XXX"),        # EV05 : statut invalide
    ev(event_id=99004, gross_weight_kg=-50),         # EV06 : poids négatif
    ev(event_id=99005, gross_weight_kg=99999),       # EV06 : poids trop grand
    ev(event_id=99006, customer_id="C_INCONNU"),     # EV07 : client inexistant
    ev(event_id=99007, vessel_id="V_INCONNU"),       # EV08 : navire inexistant
    ev(event_id=99008, yard_area=""),                # EV09 : zone vide
]
mb = m.iloc[0].to_dict()
mauvais_moves = [
    dict(mb, move_id=m.move_id.iloc[1]),                    # MV02 : identifiant en double
    dict(mb, move_id=88001, move_ts="31/31/2026"),          # MV03 : date illisible
    dict(mb, move_id=88002, equipment_id="EQ_INCONNU"),     # MV04 : équipement inexistant
    dict(mb, move_id=88003, container_id="CONT_INCONNU"),   # MV06 : conteneur inexistant
]
pd.concat([e, pd.DataFrame(mauvais_events)]).to_csv("bad/container_events.csv", index=False)
pd.concat([m, pd.DataFrame(mauvais_moves)]).to_csv("bad/yard_moves.csv", index=False)

if os.path.exists("smartport_test.db"): os.remove("smartport_test.db")
statuts = rejouer_pipeline("bad", "smartport_test.db")
print("Contrôles en échec (FAIL) sur les données défectueuses :", statuts.count("FAIL"), "(attendu : ils détectent les erreurs injectées)")

In [ ]:
tc = sqlite3.connect("smartport_test.db")
print("Lignes rejetées, par règle :")
display(pd.read_sql("SELECT rule_id, rule_description, COUNT(*) AS nb FROM etl_rejects GROUP BY 1, 2 ORDER BY 1", tc))
print("Faits chargés malgré les 13 erreurs injectées :")
display(pd.read_sql("""SELECT (SELECT COUNT(*) FROM fact_container_event) AS evenements,
                       (SELECT COUNT(*) FROM fact_yard_move) AS mouvements""", tc))
print("Contrôles qui détectent les erreurs injectées (donnée source) :")
display(pd.read_sql("SELECT check_id, layer, status, observed FROM etl_quality_checks WHERE status <> 'PASS' ORDER BY layer, check_id", tc))
print("Réconciliation (écart attendu 0) :")
display(pd.read_sql("SELECT check_id, status, observed FROM etl_quality_checks WHERE check_id IN ('Q09','Q10')", tc))

**À retenir :** les 13 lignes défectueuses sont toutes rejetées par la bonne règle, les faits restent à 9 000 événements et 4 500 mouvements, et la réconciliation reste à 0. Les contrôles placés sur la donnée source (doublons, clés orphelines) passent en `FAIL` : ils **détectent** les erreurs. Les contrôles placés sur le CORE restent à `PASS` : les lignes défectueuses ont été **rejetées** et n'ont pas pollué le DWH. Le pipeline est donc protégé contre des données sales, même si les données fournies ne le sont pas.

## 9. Python / EDA : statistiques, graphiques et interprétation
Les graphiques lisent les **tables MART** (et quelques tables du CORE) construites plus haut : c'est la couche de consommation de l'architecture.
Chaque graphique est suivi de son **interprétation** et d'une **décision possible**, comme l'exige le sujet. Les figures sont enregistrées dans le dossier `figures/` pour être insérées dans le dossier et les slides.

**Message d'ensemble :** l'activité du terminal est **très régulière**. Presque aucune différence n'est statistiquement significative (par heure, par jour de semaine, par client, par type d'événement). Les deux seules exceptions (la zone Y04 et un léger regroupement des jours chargés en mai-juin) sont signalées avec leurs réserves. Nous le disons tel quel plutôt que d'inventer des tendances.

### 9.0 Configuration des graphiques

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
                     "axes.titlesize": 11.5, "axes.titleweight": "bold"})
BLEU, ORANGE, GRIS, ROUGE, VERT = "#2E6FBA", "#E08A1E", "#8A8F98", "#C8423B", "#2F9E6E"

def sauver(nom):
    plt.tight_layout()
    plt.savefig(f"figures/{nom}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 9.1 Aperçu du jeu de données principal
C'est le point de départ proposé par le prof : dimensions, premières lignes, statistiques descriptives de `container_events.csv`.

In [ ]:
df = pd.read_csv("container_events.csv")
print(df.shape)
display(df.head())
display(df.describe(include="all").transpose())

### 9.2 Statistiques sur l'activité quotidienne
Avant de parler de « pics », on vérifie s'ils sont **réels ou dus au hasard**. L'*indice de dispersion* (variance ÷ moyenne) vaut environ 1 quand les événements arrivent au hasard (loi de Poisson). Un indice nettement supérieur à 1 indiquerait de vrais pics ou des jours particuliers.

In [ ]:
jour = q("SELECT * FROM mart_kpi_events_per_day ORDER BY full_date")
jour["full_date"] = pd.to_datetime(jour["full_date"])
moy, ecart = jour.nb_evenements.mean(), jour.nb_evenements.std()
seuil = moy + 2 * ecart
pics = jour[jour.nb_evenements > seuil]

print(f"Jours d'activité        : {len(jour)}")
print(f"Moyenne par jour        : {moy:.1f}   écart-type : {ecart:.1f}   min : {jour.nb_evenements.min()}   max : {jour.nb_evenements.max()}")
print(f"Indice de dispersion    : {jour.nb_evenements.var() / moy:.2f}   (≈ 1 = hasard pur)")
print(f"Seuil moyenne + 2 écarts-types : {seuil:.1f}  ->  {len(pics)} jours au-dessus (environ 7 attendus par pur hasard)")
print(f"95 % des jours sont <= {jour.nb_evenements.quantile(.95):.0f} événements, 99 % sont <= {jour.nb_evenements.quantile(.99):.0f}")

# Les jours chargés sont-ils répartis au hasard dans l'année ?
dans = jour.full_date.between("2026-05-01", "2026-07-10")
n_dans = pics.full_date.between("2026-05-01", "2026-07-10").sum()
print(f"\nJours chargés entre le 1er mai et le 10 juillet : {n_dans} sur {len(pics)}  (cette période = {100 * dans.mean():.0f} % du calendrier)")
print(f"Moyenne par jour : {jour[dans].nb_evenements.mean():.1f} dans la période, {jour[~dans].nb_evenements.mean():.1f} en dehors (test t : p = {stats.ttest_ind(jour[dans].nb_evenements, jour[~dans].nb_evenements).pvalue:.3f})")
print("Attention : cette période a été choisie après avoir regardé les données, donc ce p est trop optimiste.")
mensuel = [x.nb_evenements.values for _, x in jour.groupby(jour.full_date.dt.to_period("M"))]
print(f"Comparaison des 10 mois entre eux (ANOVA) : p = {stats.f_oneway(*mensuel).pvalue:.2f}  (pas de mois significativement différent)")

### 9.3 KPI « Nombre d'événements par jour »

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(jour.full_date, jour.nb_evenements, color=GRIS, lw=0.8, alpha=0.85, label="Événements par jour")
ax.plot(jour.full_date, jour.nb_evenements.rolling(7, center=True).mean(), color=BLEU, lw=2.2, label="Moyenne glissante sur 7 jours")
ax.axhline(moy, color=BLEU, ls="--", lw=1, label=f"Moyenne ({moy:.1f})")
ax.axhline(seuil, color=ROUGE, ls=":", lw=1.3, label=f"Seuil moyenne + 2 écarts-types ({seuil:.1f})")
ax.axvspan(pd.Timestamp("2026-05-01"), pd.Timestamp("2026-07-10"), color=ORANGE, alpha=0.10, label=f"1er mai - 10 juil. : {n_dans} des {len(pics)} jours chargés")
ax.scatter(pics.full_date, pics.nb_evenements, color=ROUGE, zorder=3, s=30, label=f"{len(pics)} jours au-dessus du seuil")
ax.set_title("Activité stable autour de 30 événements par jour, avec des jours chargés surtout en mai-juin")
ax.set_ylabel("Événements par jour")
ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.1))
sauver("g1_evenements_par_jour")

**Interprétation.**
- Le terminal traite en moyenne **29,9 événements par jour** (de 14 à 50). La moyenne glissante n'indique **aucune tendance** à la hausse ou à la baisse, et les 10 mois ne diffèrent pas significativement entre eux (p = 0,48).
- Jour par jour, la variabilité est celle d'un **tirage aléatoire** : l'indice de dispersion est de 1,04, et 9 jours dépassent le seuil contre environ 7 attendus par hasard. Aucun jour n'est exceptionnel en lui-même.
- **Nuance :** ces 9 jours chargés ne sont pas répartis uniformément. **7 sur 9** tombent entre le 1er mai et le 10 juillet (24 % du calendrier), et la moyenne y est un peu plus élevée (31,1 contre 29,5 ailleurs, soit environ +5 %). C'est un **signe possible de saisonnalité** en mai-juin, mais faible, et la période a été repérée après avoir regardé les données : c'est une **piste à confirmer**, pas une preuve.

**Décision possible.** Dimensionner les équipes sur un flux d'environ 30 événements par jour, avec une marge pour absorber jusqu'à **44 à 50 événements** (le 99e centile est à 44, le maximum observé à 50, le 20/06/2026). Envisager un renfort modéré de mai à début juillet, **à valider avec une deuxième année de données** avant de l'inscrire dans la planification.

### 9.4 Rythme de la journée et de la semaine

In [ ]:
heure = q("SELECT hour_key AS heure, COUNT(*) AS n FROM fact_container_event GROUP BY 1 ORDER BY 1")
jour["jour_sem"] = jour.full_date.dt.dayofweek
sem = jour.groupby("jour_sem").nb_evenements.mean()
p_h = stats.chisquare(heure.n).pvalue
p_s = stats.f_oneway(*[x.nb_evenements.values for _, x in jour.groupby("jour_sem")]).pvalue
noms = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.bar(heure.heure, heure.n, color=BLEU)
a1.axhline(heure.n.mean(), color=ORANGE, ls="--", label=f"Moyenne ({heure.n.mean():.0f})")
a1.set_title(f"Aucune heure ne se détache (test d'uniformité p = {p_h:.2f})")
a1.set_xlabel("Heure de la journée"); a1.set_ylabel("Événements sur 10 mois"); a1.set_ylim(0, heure.n.max() * 1.15); a1.legend(frameon=False)
a2.bar(noms, sem.values, color=BLEU)
a2.axhline(sem.mean(), color=ORANGE, ls="--", label=f"Moyenne ({sem.mean():.1f})")
a2.set_title(f"Aucun jour de semaine ne se détache (p = {p_s:.2f})")
a2.set_ylabel("Événements par jour (moyenne)"); a2.set_ylim(0, sem.max() * 1.25); a2.legend(frameon=False)
sauver("g2_heure_et_semaine")

**Interprétation.**
- Par heure, le volume varie de 350 (11 h) à 407 événements (21 h), soit environ **±8 %** autour de la moyenne de 375. Le test d'uniformité (p = 0,80) ne permet pas de conclure à une vraie différence : le « pic de 21 h » est compatible avec le hasard.
- Par jour de semaine, la moyenne va de 29,0 à 30,9 événements par jour (p = 0,81) : le terminal travaille **au même rythme tous les jours, week-end compris**.

**Décision possible.** Un terminal qui fonctionne 24 h sur 24 et 7 jours sur 7 a besoin d'une couverture **constante** (équipes en rotation), sans renfort ciblé sur une plage horaire. Si le comportement réel du terminal différait (par exemple des pics à l'arrivée des navires), il faudrait des horodatages issus d'un vrai système pour le voir : c'est une limite des données synthétiques.

### 9.5 KPI « MOVE par zone Yard »

In [ ]:
zone = q("SELECT area_code, parent_area, nb_move FROM mart_kpi_move_by_area ORDER BY area_code")
base = zone[zone.area_code == zone.parent_area].set_index("parent_area").nb_move
total = zone.groupby("parent_area").nb_move.sum()
sous = (total - base.reindex(total.index)).fillna(0)
p_z = stats.chisquare(zone.nb_move).pvalue

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
a1.bar(zone.area_code, zone.nb_move, color=[ORANGE if c == "Y08B" else BLEU for c in zone.area_code])
for i, v in enumerate(zone.nb_move):
    a1.text(i, v + 6, str(v), ha="center", fontsize=9)
a1.set_title(f"Zone telle que saisie : de {zone.nb_move.min()} à {zone.nb_move.max()} MOVE")
a1.set_ylabel("Événements MOVE"); a1.set_ylim(0, zone.nb_move.max() * 1.18)
a2.bar(total.index, base.reindex(total.index), color=BLEU, label="Zone principale")
a2.bar(total.index, sous, bottom=base.reindex(total.index), color=ORANGE, label="Sous-zone Y08B rattachée")
for i, v in enumerate(total.values):
    a2.text(i, v + 12, str(v), ha="center", fontsize=9, fontweight="bold")
a2.set_title("Y08B rattachée à Y08 : Y08 paraît deux fois plus chargée")
a2.set_ylim(0, total.max() * 1.18); a2.legend(frameon=False, fontsize=8, loc="upper left")
sauver("g3_move_par_zone")
print(f"Test d'uniformité des 6 zones saisies : p = {p_z:.3f}")

**Interprétation.**
- Zone par zone (graphique de gauche), les mouvements MOVE se répartissent entre **443 et 538**. **Y04 est la plus chargée** (538), environ 17 % au-dessus de la moyenne des autres zones. C'est la seule différence légèrement significative de toute l'étude (p ≈ 0,02), à prendre avec prudence : en multipliant les tests, une valeur de cet ordre peut apparaître par hasard.
- Une fois **Y08B rattachée à Y08** (graphique de droite), Y08 atteint 946 MOVE et semble deux fois plus chargée. **C'est un effet de regroupement** : Y08 additionne deux sous-zones de taille comparable aux autres zones (490 et 456). Le choix de rattacher Y08B change donc la conclusion.

**Décision possible.** Ne pas conclure que Y08 est « saturée » sans connaître la surface des zones : il faudrait rapporter les mouvements à la capacité de chaque zone. En attendant, **Y04** est la zone à surveiller en priorité. Cette lecture dépend de notre hypothèse sur Y08B, à faire valider par le propriétaire métier.

### 9.6 KPI « Volumes par navire et par service »

In [ ]:
srv = q("SELECT * FROM mart_kpi_service ORDER BY nb_evenements DESC")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.bar(srv.service_code, srv.nb_evenements, color=BLEU)
for i, (n, nav) in enumerate(zip(srv.nb_evenements, srv.nb_navires)):
    a1.text(i, n + 30, f"{n}\n({nav} navires)", ha="center", fontsize=8.5)
a1.set_title(f"{srv.service_code.iloc[0]} domine en volume ({srv.pct_evenements.iloc[0]:.0f} % des événements)")
a1.set_ylabel("Événements"); a1.set_ylim(0, srv.nb_evenements.max() * 1.2)
a2.bar(srv.service_code, srv.evenements_par_navire, color=VERT)
for i, v in enumerate(srv.evenements_par_navire):
    a2.text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
a2.set_title("... mais chaque navire pèse à peu près pareil")
a2.set_ylabel("Événements par navire"); a2.set_ylim(0, 95)
sauver("g4_volumes_par_service")

**Interprétation.**
- **GULF** génère le plus d'événements (2 469, soit 27,4 %), devant REDSEA (1 959), DJJIB (1 801), HORN (1 550) et ASIA (1 221).
- Cette hiérarchie s'explique **uniquement par le nombre de navires** de chaque service (32 pour GULF, 17 pour ASIA). Par navire, le volume est presque identique : de **71,8 à 77,2 événements** (différence non significative, p = 0,38).

**Décision possible.** Pour planifier les ressources, il faut anticiper le **nombre d'escales par service** plutôt que la taille de chaque escale : une escale GULF et une escale ASIA demandent à peu près le même travail. Le service GULF doit recevoir le plus de ressources parce qu'il a le plus d'escales, pas parce que ses escales sont plus lourdes.

### 9.7 KPI « Activité par client »

In [ ]:
cli = q("SELECT customer_name, customer_type, nb_evenements FROM mart_kpi_customer ORDER BY nb_evenements DESC")
couleur_type = {"Importer": BLEU, "Forwarder": ORANGE, "Shipping": VERT, "Exporter": ROUGE}
top = cli.head(10).iloc[::-1]
pct_top10 = 100 * cli.nb_evenements.head(10).sum() / cli.nb_evenements.sum()

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(top.customer_name, top.nb_evenements, color=[couleur_type[t] for t in top.customer_type])
ax.axvline(cli.nb_evenements.mean(), color=GRIS, ls="--", label=f"Moyenne de tous les clients ({cli.nb_evenements.mean():.0f})")
for y, v in enumerate(top.nb_evenements):
    ax.text(v + 1, y, str(v), va="center", fontsize=9)
ax.set_title(f"Aucun client ne domine : le top 10 pèse {pct_top10:.0f} % de l'activité")
ax.set_xlabel("Événements"); ax.set_xlim(0, cli.nb_evenements.max() * 1.12)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in couleur_type.values()]
ax.legend(handles + [ax.lines[0]], list(couleur_type) + [ax.lines[0].get_label()], frameon=False, fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.14))
sauver("g5_activite_par_client")
display(cli.groupby("customer_type").nb_evenements.agg(nb_clients="count", evenements="sum", moyenne_par_client="mean").round(1))

**Interprétation.**
- Les clients vont de **91 à 136 événements** pour une moyenne de 112,5. Le premier client (Customer_067) ne représente que **1,5 %** de l'activité, et les 10 premiers **14,3 %** : la clientèle est très dispersée, aucun client ne pèse assez pour mettre le terminal en dépendance.
- Par type, les moyennes sont proches (Exporter 109,9 ; Forwarder 112,8 ; Importer 113,4 ; Shipping 113,1) et les écarts ne sont pas significatifs (p = 0,66). Les importateurs pèsent le plus au total (3 290 événements) simplement parce qu'ils sont les plus nombreux (29 clients).

**Décision possible.** Le risque commercial lié à la concentration est faible. En revanche, on ne peut pas dégager un « client stratégique » avec ces données : un classement par chiffre d'affaires ou par durée de séjour serait nécessaire pour cela.

### 9.8 KPI « Poids moyen par type d'événement »

In [ ]:
typ = q("SELECT event_type, poids_moyen_kg FROM mart_kpi_event_type ORDER BY poids_moyen_kg DESC")
poids = q("SELECT fs.freight_status AS statut, f.gross_weight_kg AS poids FROM fact_container_event f JOIN dim_freight_status fs ON fs.freight_key = f.freight_key")
moy_globale = poids.poids.mean() / 1000

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})
a1.bar(typ.event_type, typ.poids_moyen_kg / 1000, color=BLEU)
a1.axhline(moy_globale, color=ORANGE, ls="--", label=f"Moyenne globale ({moy_globale:.1f} t)")
for i, v in enumerate(typ.poids_moyen_kg / 1000):
    a1.text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=8.5)
a1.set_title("Poids moyen quasi identique pour tous les types")
a1.set_ylabel("Poids brut moyen (tonnes)"); a1.set_ylim(0, 26); a1.legend(frameon=False, loc="upper right")
a1.tick_params(axis="x", labelrotation=30)
a2.boxplot([poids[poids.statut == s].poids / 1000 for s in ["FCL", "MTY"]], widths=0.5, showmeans=True)
a2.set_xticks([1, 2]); a2.set_xticklabels(["FCL (plein)", "MTY (vide)"])
a2.set_title("Un conteneur vide aussi lourd qu'un plein :\nincohérence des données")
a2.set_ylabel("Poids brut (tonnes)")
sauver("g6_poids")
print("Médiane FCL :", round(poids[poids.statut == "FCL"].poids.median()), "kg | Médiane MTY :", round(poids[poids.statut == "MTY"].poids.median()), "kg")
print("Test FCL vs MTY (Mann-Whitney) : p =", round(stats.mannwhitneyu(poids[poids.statut == "FCL"].poids, poids[poids.statut == "MTY"].poids).pvalue, 3))

**Interprétation.**
- Le poids brut moyen est de **18,6 tonnes**, et il varie très peu selon le type d'événement (de 18,4 t pour DELIVER à 18,8 t pour SEAL, p = 0,58). Un déplacement en cour ou un chargement sur navire concerne donc des conteneurs de même poids moyen.
- **Anomalie à signaler :** les conteneurs **vides (MTY)** ont exactement la même distribution de poids que les pleins (médiane de 18,6 t pour les deux, p = 0,78, et des valeurs jusqu'à 35 t). Or un conteneur vide pèse en réalité quelques tonnes. Le poids semble donc généré **indépendamment du statut**.

**Décision possible.** Ne pas utiliser le poids pour estimer la charge réelle manutentionnée tant que cette incohérence n'est pas levée. En production, on ajouterait une règle de qualité (poids d'un conteneur MTY borné à sa tare) et on interrogerait la source. C'est un point à présenter franchement au jury : il montre que nous avons contrôlé la cohérence métier, pas seulement le format.

### 9.9 KPI « Part FCL / MTY »

In [ ]:
fcl = q("SELECT event_type, pct_fcl FROM mart_kpi_event_type ORDER BY pct_fcl")
glob = q("SELECT pct_evenements FROM mart_kpi_freight WHERE freight_status = 'FCL'").iloc[0, 0]
lignes = pd.concat([pd.DataFrame({"event_type": ["TOUS"], "pct_fcl": [glob]}), fcl], ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(lignes.event_type, lignes.pct_fcl, color=[ORANGE if t == "TOUS" else BLEU for t in lignes.event_type], label="FCL (plein)")
ax.barh(lignes.event_type, 100 - lignes.pct_fcl, left=lignes.pct_fcl, color="#D5D9DF", label="MTY (vide)")
for y, v in enumerate(lignes.pct_fcl):
    ax.text(v / 2, y, f"{v:.1f} %", va="center", ha="center", color="white", fontsize=9, fontweight="bold")
ax.set_title(f"Environ {glob:.0f} % de conteneurs pleins, quel que soit le type d'événement")
ax.set_xlabel("Part des événements (%)"); ax.set_xlim(0, 100); ax.invert_yaxis()
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.14))
sauver("g7_part_fcl_mty")

**Interprétation.**
- **75,8 %** des événements concernent des conteneurs pleins (FCL) et **24,2 %** des vides (MTY), soit à peu près **3 pleins pour 1 vide**. La part reste comprise entre 74,0 % (SEAL) et 77,0 % (DELIVER) selon le type d'événement.
- Cette stabilité est surprenante : dans un vrai terminal, certaines opérations (par exemple le repositionnement de vides) concernent surtout des MTY. Ici, la répartition semble indépendante du type d'opération.

**Décision possible.** Environ un quart des mouvements porte sur des conteneurs vides : c'est un volume non négligeable à optimiser (regroupement, repositionnement) puisqu'il ne génère pas de marchandise. Une analyse par client ou par service permettrait de repérer où les vides s'accumulent.

### 9.10 Récapitulatif des 6 KPI
Tableau de synthèse à reprendre dans le dossier et dans les slides.

In [ ]:
recap = pd.DataFrame([
    ("Événements par jour",        "Moyenne",                   f"{moy:.1f} (min {jour.nb_evenements.min()}, max {jour.nb_evenements.max()})"),
    ("MOVE par zone Yard",         "Zone la plus chargée",      f"{zone.sort_values('nb_move').iloc[-1].area_code} : {zone.nb_move.max()} MOVE"),
    ("Volumes par service",        "Service le plus actif",     f"{srv.service_code.iloc[0]} : {srv.nb_evenements.iloc[0]} événements ({srv.pct_evenements.iloc[0]:.1f} %)"),
    ("Activité par client",        "Part du top 10",            f"{pct_top10:.1f} % de l'activité"),
    ("Poids moyen par événement",  "Moyenne globale",           f"{moy_globale:.1f} tonnes"),
    ("Part FCL / MTY",             "Part de conteneurs pleins", f"{glob:.1f} % FCL / {100 - glob:.1f} % MTY"),
], columns=["KPI", "Indicateur", "Valeur"])
recap

### 9.11 Exporter les figures
Le dossier `figures/` contient les 7 graphiques en PNG (haute résolution). Dans Colab, la cellule suivante télécharge une archive.

In [ ]:
print(sorted(os.listdir("figures")))
shutil.make_archive("figures", "zip", "figures")
try:
    from google.colab import files
    files.download("figures.zip")
except ImportError:
    print("figures.zip créé dans le dossier courant")

## Fin du notebook
Vous avez maintenant, de bout en bout : **sources → HDFS → RAW → STAGING → CORE → MART → KPI et graphiques**, avec contrôles qualité, rejets et métriques à chaque étape.